# Extract x1dbk1, x1dtrace, and x1dbk2 from test-data

This notebook builds on `inspiration/extraction_script.py` and focuses on the test data in `test-data/`. It will:

- create `x1d` files if they do not exist
- extract `x1dbk1`, `x1dtrace`, and `x1dbk2` using the same offsets/sizes logic

Run the cells top to bottom.

In [ ]:
import os
from pathlib import Path
from copy import copy

import numpy as np
from astropy.io import fits
import stistools as stis

# Optional: set CRDS paths if they are not already configured.
# Update CRDS_PATH to match your local cache location if needed.
CRDS_PATH = "/Users/parke/crds_cache"
if "CRDS_PATH" not in os.environ:
    os.environ["CRDS_PATH"] = CRDS_PATH
    os.environ.setdefault("CRDS_SERVER_URL", "https://hst-crds.stsci.edu")
    os.environ.setdefault("iref", f"{CRDS_PATH}/references/hst/iref/")
    os.environ.setdefault("jref", f"{CRDS_PATH}/references/hst/jref/")
    os.environ.setdefault("oref", f"{CRDS_PATH}/references/hst/oref/")
    os.environ.setdefault("lref", f"{CRDS_PATH}/references/hst/lref/")
    os.environ.setdefault("nref", f"{CRDS_PATH}/references/hst/nref/")
    os.environ.setdefault("uref", f"{CRDS_PATH}/references/hst/uref/")

In [ ]:
def find_test_data_dir() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd.parent.parent]
    for base in candidates:
        td = base / "test-data"
        if td.exists():
            return td
    raise FileNotFoundError("Could not locate test-data directory from current working dir")

TEST_DATA_DIR = find_test_data_dir()
flt_files = sorted(TEST_DATA_DIR.glob("*_flt.fits"))

print(f"Test data dir: {TEST_DATA_DIR}")
print("FLT files:")
for f in flt_files:
    print(f"- {f.name}")

In [ ]:
def get_x1dparams(header) -> dict:
    grating = str(header.get("opt_elem", "")).lower()
    min_params = dict(maxsrch=0.01, bksmode="off")

    if "g140m" in grating:
        newparams = dict(
            extrsize=19,
            bk1offst=-30, bk2offst=30,
            bk1size=20, bk2size=20,
        )
    elif "g140l" in grating:
        newparams = dict(
            extrsize=13,
            bk1offst=-30, bk2offst=30,
            bk1size=20, bk2size=20,
        )
    elif "e140m" in grating:
        newparams = dict(
            extrsize=7,
            bk1size=5, bk2size=5,
        )
    else:
        raise ValueError(f"Unsupported grating: {grating}")

    if "bk1offst" in newparams:
        assert (abs(newparams["bk1offst"]) - newparams["bk1size"] / 2) > (newparams["extrsize"] / 2) + 5
        assert (abs(newparams["bk2offst"]) - newparams["bk2size"] / 2) > (newparams["extrsize"] / 2) + 5

    return {**min_params, **newparams}


def ensure_x1d(fltfile: Path, force: bool = False) -> Path:
    x1dfile = fltfile.with_name(fltfile.name.replace("_flt", "_x1d"))
    if x1dfile.exists() and not force:
        return x1dfile

    header = fits.getheader(fltfile, 0)
    x1d_params = get_x1dparams(header)
    stis.x1d.x1d(str(fltfile), str(x1dfile), **x1d_params)
    return x1dfile


def extract_background_traces(fltfile: Path, overwrite: bool = False) -> None:
    header = fits.getheader(fltfile, 0)
    grating = str(header.get("opt_elem", "")).lower()
    x1d_params = get_x1dparams(header)

    x1dfile = ensure_x1d(fltfile, force=False)
    tracelocs = fits.getdata(x1dfile, 1)["a2center"]

    labels = ["x1dbk1", "x1dtrace", "x1dbk2"]
    mod_params = copy(x1d_params)
    mod_params["bk1size"] = mod_params["bk2size"] = 0
    mod_params["bk1offst"] = mod_params["bk2offst"] = 0
    mod_params.pop("extrsize", None)

    if "e140m" in grating:
        traceloc = tracelocs[-8]
        sets = ((traceloc, x1d_params["extrsize"], labels[1]),)
    else:
        traceloc = tracelocs[0] if np.ndim(tracelocs) else tracelocs
        y1 = traceloc + x1d_params["bk1offst"]
        yt = traceloc
        y2 = traceloc + x1d_params["bk2offst"]
        sz1 = x1d_params["bk1size"]
        szt = x1d_params["extrsize"]
        sz2 = x1d_params["bk2size"]
        sets = ((y1, sz1, labels[0]), (yt, szt, labels[1]), (y2, sz2, labels[2]))

    for y, sz, lbl in sets:
        out = fltfile.with_name(fltfile.name.replace("_flt", f"_{lbl}"))
        if out.exists() and overwrite:
            out.unlink()
        if out.exists() and not overwrite:
            continue
        stis.x1d.x1d(str(fltfile), str(out), a2center=y, extrsize=sz, **mod_params)

In [ ]:
for flt in flt_files:
    print(f"Processing {flt.name}")
    extract_background_traces(flt, overwrite=False)

print("\nOutputs:")
for f in sorted(TEST_DATA_DIR.glob("*_x1dbk1.fits")):
    print(f"- {f.name}")
for f in sorted(TEST_DATA_DIR.glob("*_x1dtrace.fits")):
    print(f"- {f.name}")
for f in sorted(TEST_DATA_DIR.glob("*_x1dbk2.fits")):
    print(f"- {f.name}")